In [0]:
# Databricks Notebook: L3_Oro.py
# ==============================================================================
# CAPA GOLD (L3) - Análisis RFM y Segmentación con K-Means
# Basado en segmentacionv3.py (líneas 200-900)
# Replica FIELMENTE la metodología, cálculos y umbrales del documento original
# ==============================================================================

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Inicializar Spark Session
spark = SparkSession.builder.appName("OroRFM").getOrCreate()

# Configuración de esquemas
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

print("="*80)
print("🏆 CAPA GOLD: Segmentación RFM con K-Means")
print("="*80)

# Crear schema Gold si no existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
print(f"✅ Schema {GOLD_SCHEMA} verificado")

In [0]:

# ==============================================================================
# PASO 1: CARGAR TABLA RFM DESDE SILVER
# ==============================================================================

print("\n📂 Paso 1: Cargando tabla RFM desde Silver...")

rfm_spark = spark.table(f"{SILVER_SCHEMA}.rfm_features")
rfm_logistica = rfm_spark.toPandas()

print(f"   ✅ Cargados {len(rfm_logistica):,} clientes")
print(f"\n   Primeras 5 filas:")
print(rfm_logistica.head())

In [0]:

# ==============================================================================
# PASO 2: ANÁLISIS DE CORRELACIÓN Y SELECCIÓN DE FEATURES
# ==============================================================================

print("\n📊 Paso 2: Análisis de correlación de features...")

# Features iniciales (con Antigüedad)
features_list_initial = [
    'Recencia',
    'Frecuencia',
    'Monetario',
    'Antiguedad',
    'Amplitud_Categorias',
    'Total_Articulos',
    'Avg_Peso_g',
    'Avg_Volumen_cm3',
    'Avg_Installments'
]

# Preparar datos para análisis de correlación
data_to_analyze = rfm_logistica[features_list_initial].fillna(0)
data_log_analyze = np.log1p(data_to_analyze)
scaler_analyze = StandardScaler()
data_scaled_analyze = scaler_analyze.fit_transform(data_log_analyze)

# Calcular matriz de correlación
data_scaled_df = pd.DataFrame(data_scaled_analyze, columns=features_list_initial)
corr_matrix = data_scaled_df.corr()

print("\n   🔍 Matriz de correlación calculada")
print(f"\n   Correlación Recencia vs Antiguedad: {corr_matrix.loc['Recencia', 'Antiguedad']:.3f}")

# DECISIÓN: Eliminar 'Antiguedad' por alta correlación con 'Recencia' (> 0.9)
print("\n   ⚠️  NOTA: Se elimina 'Antiguedad' por alta correlación con 'Recencia'")

# Features finales (sin Antigüedad) - EXACTAMENTE como en segmentacionv3.py
features_list_final = [
    'Recencia',
    'Frecuencia',
    'Monetario',
    'Amplitud_Categorias',
    'Total_Articulos',
    'Avg_Peso_g',
    'Avg_Volumen_cm3',
    'Avg_Installments'
]

print(f"\n   ✅ Features finales seleccionadas ({len(features_list_final)}):")
for i, feat in enumerate(features_list_final, 1):
    print(f"      {i}. {feat}")

In [0]:

# ==============================================================================
# PASO 3: PREPROCESAMIENTO (LOG + SCALING)
# ==============================================================================

print("\n🔄 Paso 3: Preprocesamiento de datos...")

# Separar datos a procesar
data_to_process_final = rfm_logistica[features_list_final].copy()
data_to_process_final = data_to_process_final.fillna(0)

print(f"   Datos a procesar: {data_to_process_final.shape}")

# 1. Transformación Logarítmica (para manejar outliers/sesgo)
print("   Aplicando transformación logarítmica: log(1 + x)")
data_log_final = np.log1p(data_to_process_final)

# 2. Escalado (StandardScaler para que todas las features tengan misma importancia)
print("   Aplicando StandardScaler (mean=0, std=1)")
scaler_final = StandardScaler()
data_scaled_final = scaler_final.fit_transform(data_log_final)

print(f"   ✅ Datos preprocesados: {data_scaled_final.shape}")

In [0]:
# ==============================================================================
# PASO 4: MÉTODO DEL CODO Y SILUETA (Selección de K)
# ==============================================================================

print("\n📈 Paso 4: Determinando número óptimo de clusters (K)...")

K_range = range(2, 11)
inertia_list = []
silhouette_list = []

print("   Calculando Inercia y Silueta para K=2 a 10...")

metrics_records = []  # ← NUEVO (para guardar en tabla Gold)

for k in K_range:
    # Entrenar K-Means
    kmeans_model = KMeans(
        n_clusters=k,
        init='k-means++',
        n_init=10,
        max_iter=300,
        random_state=42
    )
    kmeans_model.fit(data_scaled_final)

    # Calcular valores
    inertia = kmeans_model.inertia_
    labels = kmeans_model.labels_
    silhouette = silhouette_score(data_scaled_final, labels)

    inertia_list.append(inertia)
    silhouette_list.append(silhouette)

    print(f"      K={k}: Inercia={inertia:.2f}, Silueta={silhouette:.4f}")

    # Guardar registro para tabla Gold
    metrics_records.append({
        "K": k,
        "Inercia": float(inertia),
        "Silueta": float(silhouette),
        "Fecha_Proceso": pd.Timestamp.now()
    })

# Convertir a Spark y guardar tabla GOLD
metrics_df = pd.DataFrame(metrics_records)
metrics_spark_df = spark.createDataFrame(metrics_df)

metrics_spark_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{GOLD_SCHEMA}.kmeans_metrics")

print(f"\n   💾 Tabla de métricas guardada en {GOLD_SCHEMA}.kmeans_metrics")
print(metrics_df.head())

# Seleccionar mejor K por Silueta
best_k_index = np.argmax(silhouette_list)
best_k_silhouette = list(K_range)[best_k_index]
best_silhouette_score = silhouette_list[best_k_index]

print(f"\n   📊 Mejor K según Silueta: K={best_k_silhouette} (Score: {best_silhouette_score:.4f})")

# DECISIÓN: Usar K=5 (criterio de negocio)
optimal_k = 5
print(f"\n   ✅ K seleccionado para modelo final: {optimal_k} (criterio de negocio)")


In [0]:

# ==============================================================================
# PASO 5: ENTRENAMIENTO DEL MODELO FINAL
# ==============================================================================

print(f"\n🤖 Paso 5: Entrenando modelo K-Means con K={optimal_k}...")

# Entrenar modelo final
kmeans_final = KMeans(
    n_clusters=optimal_k,
    init='k-means++',
    n_init=10,
    max_iter=300,
    random_state=42
)

kmeans_final.fit(data_scaled_final)

# Obtener segmentos
segmentos = kmeans_final.labels_

# Añadir segmentos a la tabla RFM
rfm_logistica['Segmento'] = segmentos

# Calcular silueta del modelo final
silhouette_final = silhouette_score(data_scaled_final, segmentos)

print(f"   ✅ Modelo entrenado")
print(f"   📊 Silhouette Score: {silhouette_final:.4f}")
print(f"   📊 Inertia: {kmeans_final.inertia_:.2f}")

# Distribución de segmentos
print(f"\n   📊 Distribución de segmentos:")
segment_dist = rfm_logistica['Segmento'].value_counts().sort_index()
for seg_id, count in segment_dist.items():
    pct = (count / len(rfm_logistica)) * 100
    print(f"      Segmento {seg_id}: {count:,} clientes ({pct:.1f}%)")

In [0]:

# ==============================================================================
# PASO 6: PERFIL DE SEGMENTOS
# ==============================================================================

print("\n📊 Paso 6: Calculando perfil promedio de cada segmento...")

segment_profile = rfm_logistica.groupby('Segmento')[features_list_final].mean()

print("\n   Perfil Promedio por Segmento:")
print(segment_profile)

# Dimensionamiento financiero (análisis adicional)
print("\n💰 Análisis Financiero de los Segmentos:")

cols_analisis = ['Monetario', 'Avg_Installments', 'Frecuencia', 'Recencia', 'Avg_Peso_g']
perfil_financiero = rfm_logistica.groupby('Segmento')[cols_analisis].mean()
perfil_financiero_sorted = perfil_financiero.sort_values(by='Monetario', ascending=False)

print(perfil_financiero_sorted)

In [0]:


# ==============================================================================
# PASO 7: ASIGNAR NOMBRES DE NEGOCIO A SEGMENTOS
# ==============================================================================

print("\n🏷️  Paso 7: Asignando nombres de negocio a segmentos...")

# Mapeo de segmentos (basado en análisis de segmentacionv3.py)
# NOTA: Estos IDs pueden variar según los datos, ajustar según perfil
segment_kmeans_map = {
    2: 'Premium Leal',      # Mayor Frecuencia y Amplitud
    1: 'Ballena',           # Mayor Monetario
    0: 'Voluminoso',        # Mayor Peso/Volumen
    4: 'Nuevo',             # Menor Recencia
    3: 'Bajo Costo'         # Menor Monetario
}

rfm_logistica['Segmento_Nombre'] = rfm_logistica['Segmento'].map(segment_kmeans_map)

print("   ✅ Nombres asignados:")
for seg_id, nombre in segment_kmeans_map.items():
    count = (rfm_logistica['Segmento'] == seg_id).sum()
    print(f"      Segmento {seg_id} → {nombre} ({count:,} clientes)")

In [0]:
# ==============================================================================
# PASO 8: CALCULAR RFM SCORES (Scoring Tradicional)
# ==============================================================================

print("\n🎯 Paso 8: Calculando RFM Scores tradicionales...")

# Crear tabla de scores
rfm_scores = rfm_logistica[['customer_unique_id', 'Recencia', 'Frecuencia', 'Monetario']].copy()

# R_Score (Recencia): Menor recencia = Mayor score
rfm_scores['R_Score'] = pd.qcut(rfm_scores['Recencia'], 5, labels=[5, 4, 3, 2, 1], duplicates='drop').astype(int)

# F_Score (Frecuencia): Método manual por distribución sesgada
def score_frecuencia(f):
    if f > 2:
        return 5  # Premium
    elif f == 2:
        return 3  # Recurrente
    else:
        return 1  # Ocasional

rfm_scores['F_Score'] = rfm_scores['Frecuencia'].apply(score_frecuencia)

# M_Score (Monetario): Mayor gasto = Mayor score
rfm_scores['M_Score'] = pd.qcut(rfm_scores['Monetario'], 5, labels=[1, 2, 3, 4, 5], duplicates='drop').astype(int)

# RFM Score concatenado
rfm_scores['RFM_Score'] = (
    rfm_scores['R_Score'].astype(str) + 
    rfm_scores['F_Score'].astype(str) + 
    rfm_scores['M_Score'].astype(str)
)

print(f"   ✅ RFM Scores calculados para {len(rfm_scores):,} clientes")
print(f"\n   Muestra de scores:")
print(rfm_scores.head())


In [0]:

# ==============================================================================
# PASO 9: GUARDAR TABLAS EN GOLD
# ==============================================================================

print(f"\n💾 Paso 9: Guardando tablas en {GOLD_SCHEMA}...")

# Tabla 1: Customer Segments (Principal)
customer_segments = rfm_logistica[[
    'customer_unique_id', 
    'Segmento', 
    'Segmento_Nombre',
    'Recencia', 
    'Frecuencia', 
    'Monetario',
    'Amplitud_Categorias',
    'Total_Articulos',
    'Avg_Peso_g',
    'Avg_Volumen_cm3',
    'Avg_Installments'
]]

# Añadir scores RFM
customer_segments = customer_segments.merge(
    rfm_scores[['customer_unique_id', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score']], 
    on='customer_unique_id', 
    how='left'
)

# Convertir a Spark y guardar
customer_segments_spark = spark.createDataFrame(customer_segments)
customer_segments_spark.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{GOLD_SCHEMA}.customer_segments")

print(f"   ✅ Tabla 1: {GOLD_SCHEMA}.customer_segments ({len(customer_segments):,} registros)")

# Tabla 2: Segment Profiles (Agregado)
segment_profiles = rfm_logistica.groupby(['Segmento', 'Segmento_Nombre']).agg({
    'customer_unique_id': 'count',
    'Recencia': 'mean',
    'Frecuencia': 'mean',
    'Monetario': ['mean', 'sum'],
    'Amplitud_Categorias': 'mean',
    'Avg_Peso_g': 'mean',
    'Avg_Volumen_cm3': 'mean',
    'Avg_Installments': 'mean'
}).reset_index()

# Aplanar columnas multi-nivel
segment_profiles.columns = [
    'Segmento', 'Segmento_Nombre', 'Customer_Count',
    'Avg_Recencia', 'Avg_Frecuencia', 'Avg_Monetario', 'Total_Revenue',
    'Avg_Amplitud_Categorias', 'Avg_Peso_g', 'Avg_Volumen_cm3', 'Avg_Installments'
]

# Calcular porcentajes
total_customers = len(rfm_logistica)
total_revenue = rfm_logistica['Monetario'].sum()

segment_profiles['Pct_Customers'] = (segment_profiles['Customer_Count'] / total_customers) * 100
segment_profiles['Pct_Revenue'] = (segment_profiles['Total_Revenue'] / total_revenue) * 100
segment_profiles['Value_Index'] = segment_profiles['Pct_Revenue'] / segment_profiles['Pct_Customers']

# Convertir a Spark y guardar
segment_profiles_spark = spark.createDataFrame(segment_profiles)
segment_profiles_spark.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{GOLD_SCHEMA}.segment_profiles")

print(f"   ✅ Tabla 2: {GOLD_SCHEMA}.segment_profiles ({len(segment_profiles)} segmentos)")

# Tabla 3: RFM Scores
rfm_scores_spark = spark.createDataFrame(rfm_scores)
rfm_scores_spark.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{GOLD_SCHEMA}.rfm_scores")

print(f"   ✅ Tabla 3: {GOLD_SCHEMA}.rfm_scores ({len(rfm_scores):,} registros)")

In [0]:


# ==============================================================================
# PASO 10: REPORTE FINAL
# ==============================================================================

print("\n" + "="*80)
print("📊 REPORTE DE SEGMENTACIÓN")
print("="*80)
print(f"Total Clientes Analizados: {total_customers:,}")
print(f"Total Ingresos: R$ {total_revenue:,.2f}")
print(f"Silhouette Score: {silhouette_final:.4f}")
print(f"Número de Segmentos: {optimal_k}")
print("-"*80)

print("\nDistribución de Valor por Segmento:")
print(segment_profiles[['Segmento_Nombre', 'Customer_Count', 'Pct_Customers', 
                        'Pct_Revenue', 'Value_Index']].to_string(index=False))

print("\n" + "="*80)
print("✅ CAPA GOLD COMPLETADA")
print("="*80)
print(f"Tablas creadas en {GOLD_SCHEMA}:")
print(f"  1. customer_segments ({len(customer_segments):,} clientes)")
print(f"  2. segment_profiles ({len(segment_profiles)} segmentos)")
print(f"  3. rfm_scores ({len(rfm_scores):,} clientes)")
print("="*80)

# Verificar tablas creadas
spark.sql(f"SHOW TABLES IN {GOLD_SCHEMA}").show()

print("\n*** PROCESO L3 (GOLD) FINALIZADO EXITOSAMENTE ***")